[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-4/lab-4.1-ring-all-gather.ipynb)

# LAB·4.1 · Ring all-gather from remote DMAs

**Hardware:** the algorithm runs anywhere (8 simulated devices on CPU); the Pallas remote-DMA version needs a real multi-chip TPU slice.

A collective is not a primitive; it is a kernel someone wrote. Today you write all-gather as what it physically is on a TPU pod: N-1 hops around a ring, each device pushing one shard to its neighbor while keeping compute free. First as `ppermute` steps (the algorithm, runnable anywhere), then the same loop as Pallas remote DMAs (the mechanism, TPU slice only).

In [ ]:
import os
# simulate 8 devices when no real multi-chip slice is attached; must run before jax imports
if "COLAB_TPU_ADDR" not in os.environ:
    os.environ.setdefault("XLA_FLAGS", "--xla_force_host_platform_device_count=8")

import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.sharding import Mesh, PartitionSpec as P
from jax.experimental.shard_map import shard_map
from functools import partial

devs = jax.devices()
print(jax.__version__, len(devs), "devices:", devs[0].platform)
ON_TPU = devs[0].platform == "tpu"

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


In [ ]:
# the ring algorithm: after step t, every device holds t+1 shards
mesh = Mesh(np.array(devs), ("x",))
AXIS = len(devs)

@partial(shard_map, mesh=mesh, in_specs=P("x", None), out_specs=P("x", None))
def ring_all_gather(shard):
    n = AXIS  # static mesh size from the closure
    idx = jax.lax.axis_index("x")
    out = jnp.zeros((n, *shard.shape[1:]), shard.dtype)
    out = out.at[idx].set(shard[0])
    recv = shard[0]
    def hop(t, carry):
        out, recv = carry
        recv = jax.lax.ppermute(recv, "x", [(i, (i + 1) % n) for i in range(n)])
        src = (idx - t - 1) % n
        return out.at[src].set(recv), recv
    out, _ = jax.lax.fori_loop(0, n - 1, hop, (out, recv))
    return out[None]

x = jax.random.normal(jax.random.key(0), (AXIS, 4, 128))
got = ring_all_gather(x).reshape(AXIS, AXIS, 4, 128)
want = jnp.broadcast_to(x[None], (AXIS, AXIS, 4, 128))
check("ring all-gather == replicated truth", got, want, tol=0)
print(f"validated over {AXIS} devices: every device reconstructed all {AXIS} shards")

## The same loop as a Pallas kernel (real TPU slice)

On a real slice the hop is not `ppermute`; it is `make_async_remote_copy`: your chip *pushes* its buffer into the neighbor's VMEM and signals a semaphore, while the MXU stays free for whatever you overlap. The [distributed Pallas tutorial](https://docs.jax.dev/en/latest/pallas/tpu/distributed.html) builds exactly this kernel; on a v5e-8 or larger, reproduce it, validate bitwise against `jax.lax.all_gather`, then do the two exercises that teach more than any reading:

1. Profile it: Find the remote copy of hop t overlapping the arrival of hop t-1.
2. Break it: Reorder one semaphore wait, run under interpret mode, and look at what a deadlock actually looks like. You will meet this failure again; better here than in a training run.

Gate criterion: bitwise match against the collective, overlap visible in the profile.